# 4.2 Tool Selection: Does a Tool Graph Choose the Right Tools Better Than Embedding Search?

*Formerly notebook 26.*

> **Outputs in this copy come from the confidence run** (`SMASH_MODE=batch`): TaskBench 1,500 samples and ToolBench
> 1,000 queries, 6,500 calls through the Batch API, $0.61. Task 4.2. They are reported in `RESULTS-REPORT.md` and the
> v2 paper.

**Presentation map:** Task 4.2 (tool selection over a tool graph), paper v2 §5 / §5a.

## Protocol (frozen before the test run; PLAN §5b)

The task is tested in two parts: **Part 1, TaskBench** (small tool sets with a published tool graph and gold tool
DAGs) and **Part 2, ToolBench** (retrieval from ~47k APIs, the large-tool-set case). Part 2 was added after the Part 1
test run, on the coordinator's request, and frozen before its own test run.

### Part 1: TaskBench

**Research question.** With many tools, does a **tool graph** (tools linked by type-compatible input → output edges)
choose the right tools for a task better than **embedding search over the tool descriptions**?

**Hypothesis.** Graph traversal from the task finds tools that embedding similarity misses (a tool in the middle of a
chain is rarely named in the request) and tells the planner which tools can feed which, so it raises node-F1 over
embedding top-k at the same candidate budget, and raises edge-F1. Putting all tools in the prompt is expected to be
at least as accurate as either, at about 2–3× the prompt tokens.

**Data.** TaskBench (Shen et al., NeurIPS 2024 D&B; Microsoft JARVIS, Hugging Face `microsoft/Taskbench` at a pinned
revision, **MIT licence**). Three domains, each with an explicit tool graph and a gold tool-invocation graph (nodes
and links) per sample: Hugging Face tasks (23 tools, 225 type-labelled edges), multimedia tools (40 tools, 449 edges),
daily-life APIs (40 tools; the published graph is **complete**, 1,560 edges of type `complete`). The `data.json`
file of each domain holds the human-verified samples (7,458 / 5,555 / 4,318). **n = 500 samples per domain, 1,500 in
all**, drawn with `random.Random(7)` from the samples with at least one gold tool; the test run uses the first 25 per
domain of the same order.

**Conditions** (one planner prompt; the planner returns the tool nodes in order and the links between them):
- **(a) All tools** — every tool of the domain in the prompt, each with its description and its input/output types
  (daily-life: its typed parameters).
- **(b) Embedding top-k** — the k = 8 tools whose description (same text as in (a), so the types are included) is
  most similar to the request (`text-embedding-3-small`, cosine).
- **(c) Tool graph** — seed with the 3 tools most similar to the request (same embeddings); expand to the tools joined
  to a seed by a type-compatible edge (either direction); rank the expansion by the number of seeds it is joined to,
  then by similarity; keep 8 tools in all. The prompt lists the same tool texts **plus the edges of the tool graph
  among the 8 candidates** (which tool's output can be the input of which). k = 8 is the largest gold tool count in
  the data, so a perfect retriever can always include every gold tool.

Same information: (b) gets the input/output types of every candidate in text, from which the edges follow; (c)
gets them as an explicit graph. (a) gets every tool.

**Primary metric.** Node-F1 per sample (set of predicted tool names vs. gold tool names), averaged over samples;
the verdict is **(c) − (b)** pooled over the three domains, paired 95% bootstrap interval (5,000 resamples, seed 7).

**Secondary.** Edge-F1 per sample (on samples with ≥ 1 gold link); exact match of the node set; candidate recall (the
share of gold tools among the tools shown); TaskBench-style micro node-F1 and link-F1; prompt tokens per call;
(c) − (a) and (b) − (a); results per domain and per structure (single node, chain, DAG).

**Analysis.** Paired bootstrap over samples for every difference; the test run checks that the interval on the primary
difference is narrower than ±5 points at full n (PLAN §5b rule 3).

**Threats.**
- **TaskBench samples were generated from its own tool graph** (a subgraph was sampled, then an LLM wrote the request),
  so every gold link is an edge of the graph: the graph result is an **upper bound** for a graph built independently
  of the tasks (PLAN §5b rule 5).
- **Small tool sets.** 23–40 tools fit in one prompt, so this measures selection quality and prompt cost, not retrieval
  at scale (published note: TaskBench is too small for a retrieval question).
- **Dense graphs.** The tool graphs are dense (mean undirected degree 15 of 22 neighbours for Hugging Face, 18 of 39
  for multimedia; daily-life is complete), so the expansion rarely excludes a tool; the ranking by seed connections and
  the explicit edges carry most of the graph's effect. On daily-life the graph has no information beyond (b); it is a
  control.
- One model (`gpt-6-luna`, no reasoning), one prompt, one k, one run.

### Part 2: ToolBench (large tool set)

**Research question.** When the agent must pick the right APIs out of tens of thousands, does traversing a tool
graph give a better candidate list for the LLM than embedding search over the API descriptions?

**Hypothesis.** The APIs a query needs often share a tool (and a category) with the API that matches the query best,
but are described in words the query does not use; expanding from the best-matching APIs to their siblings in the
tool hierarchy puts more gold APIs in front of the reranker, so recall@5 after the same LLM rerank rises.

**Data.** ToolBench (Qin et al., ICLR 2024; OpenBMB, the README states the dataset is "distributed under Apache License
2.0"; code also Apache 2.0). The official download links (Google Drive id in the README, Tsinghua Cloud) return 404 /
"link does not exist" on 2026-10-27, so the official `data.zip` is read from a public re-upload on Hugging Face
(`nullwwg/toolbench-data`, pinned revision) with HTTP range requests: only the test queries and the tool JSONs are
extracted (about 170 MB of the 1.76 GB archive), no code. **Queries:** the six ToolBench test sets (G1-instruction,
G1-category, G1-tool, G2-instruction, G2-category, G3-instruction; 1,100 queries), each with its gold `relevant APIs`.
**n = 1,000 queries** drawn with `random.Random(7)`; the test run uses the first 25. **Corpus:** every API of ToolBench's
tool environment (`toolenv/tools`: 10,657 tool files in 50 categories, 49,937 API entries, deduplicated by
(tool name, API name)). This is larger than the 16,464 APIs ToolBench reports for instruction generation (that filtered
list is not distributed) and larger than the 10,439-API G1 retrieval corpus, which covers G1 only.

**API text** (embedding and prompts, same in both conditions): category, tool name, tool description (≤ 200
characters), API name, API description (≤ 300 characters), parameter names and types.

**Conditions** (the same LLM reranker; candidate lists of 50 APIs):
- **(b) Embedding top-50 + rerank** — the 50 APIs most similar to the query (`text-embedding-3-small`, cosine).
- **(c) Tool graph + rerank** — the graph is **ToolBench's own category → tool → API hierarchy** (no LLM
  construction). Seed with the 10 APIs most similar to the query; move up to their tools and down to the tools' APIs;
  per tool keep up to 8 APIs ranked by similarity, tools in the order their first seed appears; fill to 50 with the
  embedding order if the expansion is shorter.

**Reranker.** One prompt for both: the 50 candidates grouped by tool (category and tool description shown once per
tool), numbered; the LLM returns the numbers of the APIs needed by the query, most relevant first (at most 10); the
ranking is completed with the remaining candidates in candidate order.

**Primary metric.** Recall@5 after the rerank (gold APIs in the top 5 / gold APIs), averaged over queries; verdict on
**(c) − (b)**, paired 95% bootstrap.

**Secondary.** NDCG@5 (binary relevance), recall@10, candidate recall@50, recall@5 of each candidate order without the
rerank, prompt tokens per rerank call, results per test set.

**Threats.**
- **The queries were generated from the hierarchy**: ToolBench sampled the APIs of a G1 query from one tool, of a G2
  query from one category, of a G3 query from one collection, and an LLM wrote the query. Sibling expansion matches
  that generator, so the graph result is an **upper bound** for a graph built independently of the queries.
- **Mirror.** The archive comes from a third-party re-upload of the official `data.zip` (no official checksum to
  compare). The API documentation was crawled from RapidAPI; its terms were not assessed beyond the dataset licence.
- **Name collisions.** 184 tool names occur in more than one tool; about 1% of the gold APIs (28 of 2,645) do not match
  any (tool, API) name in the environment and are unreachable for both conditions.
- One model, one reranker prompt, one candidate size; the embedding baseline is `text-embedding-3-small`, not the
  stronger fine-tuned retrievers of ToolRet / TGR.

ToolRet (PLAN §5a) is not used: its Hugging Face datasets declare no licence and re-distribute 35 source benchmarks
under mixed licences; only its code repository is Apache 2.0. ToolBench replaces it.

**Protocol changes after the test run.**
1. *(Part 1) Link definition for daily-life APIs.* The first test run defined a link as "the output of the source is an input
   of the target" in all domains. TaskBench's own inference prompt defines daily-life links as **temporal** order
   (the order in which the APIs are invoked) and Hugging Face / multimedia links as resource dependencies; under the
   data-flow wording, independent daily-life calls were left unlinked (edge-F1 of (a) 14.5%), and listing the complete
   graph in (c) pushed the planner to link them, an artefact in favour of (c). The system prompt (all three conditions)
   and the graph header of (c) now use TaskBench's definition per domain. Node selection, k and the seeds are
   unchanged; the Hugging Face and multimedia requests are unchanged.

In [ ]:
# Notebooks live in examples/<task group>/; data/, smash_batch.py and the .venv-* environments stay in examples/.
# Every path below is relative to examples/, so run from there whether the notebook is opened in its own folder or not.
import os, sys
if not os.path.exists("smash_batch.py") and os.path.exists(os.path.join("..", "smash_batch.py")):
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [1]:
import json, os, re, random, urllib.request
from collections import Counter, defaultdict
import numpy as np
from pydantic import BaseModel, Field
from smash_batch import LLM, mode_from_env, paired_boot

MODE = mode_from_env()               # replay (default) | test (small live run) | batch (full run via Batch API)
TEST = MODE == "test"
BASE = "data/benchmarks/taskbench"
DOMAINS = ("huggingface", "multimedia", "dailylifeapis")
REV = "d12764a8fa650f49f50507a02c3e707c71c67621"     # pinned revision of huggingface.co/datasets/microsoft/Taskbench
URL = f"https://huggingface.co/datasets/microsoft/Taskbench/resolve/{REV}"

# Download only if missing (data files only; no code from the dataset repository is used). Licence: MIT.
for dom in DOMAINS:
    os.makedirs(f"{BASE}/{dom}", exist_ok=True)
    for f in ("data.json", "graph_desc.json", "tool_desc.json"):
        p = f"{BASE}/{dom}/{f}"
        if not os.path.exists(p):
            urllib.request.urlretrieve(f"{URL}/data_{dom}/{f}", p)
            print("downloaded", p)

llm = LLM(f"{BASE}/cache" + ("_test" if TEST else ""))
N_PER = 500                          # samples per domain (full run)
N_RUN = 25 if TEST else N_PER        # test mode: the first 25 of the same seeded order
K, N_SEED = 8, 3                     # candidate tools shown in (b) and (c); embedding seeds of (c)
print("MODE =", MODE)

MODE = batch


## Samples, tools and the tool graph of each domain

In [2]:
def tool_text(t):
    # One line per tool, used for the embedding and in every prompt (same text in all three conditions).
    if "parameters" in t:
        ps = "; ".join(f"{p['name']} ({p['type']})" for p in t["parameters"])
        return f"{t['id']}: {t['desc']} Parameters: {ps or 'none'}."
    return (f"{t['id']}: {t['desc']} Input types: {', '.join(t['input-type'])}. "
            f"Output types: {', '.join(t['output-type'])}.")

DATA = {}
for dom in DOMAINS:
    rows = [json.loads(l) for l in open(f"{BASE}/{dom}/data.json")]
    pool = [r for r in rows if r["task_nodes"]]
    sample = random.Random(7).sample(pool, N_PER)[:N_RUN]
    g = json.load(open(f"{BASE}/{dom}/graph_desc.json"))
    names = [t["id"] for t in g["nodes"]]
    out_e = defaultdict(dict)                         # source -> {target: edge type}
    for l in g["links"]:
        out_e[l["source"]][l["target"]] = l.get("type", "")
    adj = defaultdict(set)                            # undirected neighbours (for the expansion)
    for s, ts in out_e.items():
        for t in ts:
            adj[s].add(t); adj[t].add(s)
    DATA[dom] = {"samples": sample, "tools": g["nodes"], "names": names, "text": {t["id"]: tool_text(t) for t in g["nodes"]},
                 "out": out_e, "adj": adj, "canon": {n.lower().strip(): n for n in names},
                 "dep": "temporal" if "parameters" in g["nodes"][0] else "resource"}
    deg = np.mean([len(adj[n]) for n in names])
    print(f"{dom}: {len(rows):,} human-verified samples ({len(pool):,} with >= 1 tool), {len(sample)} used | "
          f"{len(names)} tools, {len(g['links']):,} edges, mean undirected degree {deg:.1f} | "
          f"structure {dict(Counter(r['type'] for r in sample))} | gold tools per sample "
          f"{np.mean([len(r['task_nodes']) for r in sample]):.2f}")

huggingface: 7,458 human-verified samples (7,458 with >= 1 tool), 500 used | 23 tools, 225 edges, mean undirected degree 15.4 | structure {'chain': 250, 'dag': 56, 'single': 194} | gold tools per sample 2.68
multimedia: 5,555 human-verified samples (5,554 with >= 1 tool), 500 used | 40 tools, 449 edges, mean undirected degree 18.2 | structure {'dag': 50, 'chain': 263, 'single': 187} | gold tools per sample 2.82


dailylifeapis: 4,318 human-verified samples (4,318 with >= 1 tool), 500 used | 40 tools, 1,560 edges, mean undirected degree 39.0 | structure {'single': 141, 'chain': 330, 'dag': 29} | gold tools per sample 3.11


## Candidates for (b) embedding top-k and (c) tool graph

Embeddings of the tool texts and of the requests (`text-embedding-3-small`). (b) takes the 8 most similar tools.
(c) takes the 3 most similar tools as seeds, expands along the graph edges, ranks the expansion by the number of
seeds a tool is joined to and then by similarity, and keeps 8 tools.

In [3]:
for dom, d in DATA.items():
    d["T"] = llm.embed(f"nb26_{dom}_tools", [d["text"][n] for n in d["names"]])
    d["Q"] = llm.embed(f"nb26_{dom}_requests", [r["user_request"] for r in d["samples"]])

def cand_embed(d, i):
    s = d["T"] @ d["Q"][i]
    return [d["names"][j] for j in np.argsort(-s)[:K]]

def cand_graph(d, i):
    s = d["T"] @ d["Q"][i]
    order = [d["names"][j] for j in np.argsort(-s)]
    seeds = order[:N_SEED]
    rank = {n: r for r, n in enumerate(order)}
    links = {n: sum(n in d["adj"][x] for x in seeds) for n in order if n not in seeds}
    exp = sorted((n for n, c in links.items() if c > 0), key=lambda n: (-links[n], rank[n]))
    return seeds + exp[:K - N_SEED]

def gold_nodes(r):
    return {x["task"] for x in r["task_nodes"]}

def gold_links(r):
    return {(l["source"], l["target"]) for l in r["task_links"]}

for dom, d in DATA.items():
    d["cand"] = {"all": [list(d["names"])] * len(d["samples"]),
                 "embed": [cand_embed(d, i) for i in range(len(d["samples"]))],
                 "graph": [cand_graph(d, i) for i in range(len(d["samples"]))]}
    rec = {m: np.mean([len(gold_nodes(r) & set(c)) / len(gold_nodes(r)) for r, c in zip(d["samples"], d["cand"][m])])
           for m in ("embed", "graph")}
    same = np.mean([set(a) == set(b) for a, b in zip(d["cand"]["embed"], d["cand"]["graph"])])
    print(f"{dom}: candidate recall of gold tools  embed top-{K} {rec['embed']:.1%}  |  graph {rec['graph']:.1%}  "
          f"|  identical candidate sets {same:.0%}  |  graph candidates per sample "
          f"{np.mean([len(c) for c in d['cand']['graph']]):.1f}")

huggingface: candidate recall of gold tools  embed top-8 85.6%  |  graph 83.1%  |  identical candidate sets 21%  |  graph candidates per sample 8.0
multimedia: candidate recall of gold tools  embed top-8 86.7%  |  graph 85.5%  |  identical candidate sets 20%  |  graph candidates per sample 8.0
dailylifeapis: candidate recall of gold tools  embed top-8 89.6%  |  graph 89.6%  |  identical candidate sets 100%  |  graph candidates per sample 8.0


## The planner (one Batch job per condition)

The same system prompt for all three conditions. The user message lists the candidate tools (all tools for (a)); for
(c) it adds the tool-graph edges among the candidates. The planner returns the tool nodes and the links between them
(TaskBench's `task_nodes` / `task_links`, without arguments: parameter prediction is not measured).

In [4]:
class Link(BaseModel):
    source: str
    target: str

class Plan(BaseModel):
    nodes: list[str] = Field(description="names of the tools to call, in execution order")
    links: list[Link] = Field(description="(source, target) pairs where the output of source is an input of target")

# TaskBench's own link definitions (its inference.py): resource dependencies for the Hugging Face and multimedia
# tools (typed inputs/outputs), temporal order for the daily-life APIs (typed parameters, complete graph).
LINK_DEF = {"resource": "'links' are pairs (source, target) where the output of the source tool is used as an input "
                        "of the target tool.",
            "temporal": "'links' are pairs (source, target) where the source tool is invoked before the target tool: "
                        "they give the order in which the tools are called."}

def sys_prompt(d):
    return ("You plan tool calls for a user request. From the tools listed, choose the tools needed to fulfil the "
            "request and give the tool invocation graph: 'nodes' are the tool names in execution order; "
            + LINK_DEF[d["dep"]] + " Use tool names exactly as listed, and only the tools the request needs.")

def graph_lines(d, cands):
    cs, out = set(cands), []
    for s in cands:
        ts = [f"{t} [{d['out'][s][t]}]" if d["out"][s][t] not in ("", "complete") else t
              for t in d["out"].get(s, {}) if t in cs]
        if ts:
            out.append(f"- {s} -> " + ", ".join(ts))
    return out

def prompt(d, i, m):
    cands = d["cand"][m][i]
    head = ("Tools (name: description, input and output types):" if "parameters" not in d["tools"][0]
            else "Tools (name: description, parameters):")
    msg = head + "\n" + "\n".join(f"- {d['text'][n]}" for n in cands)
    if m == "graph":
        gl = graph_lines(d, cands)
        head = ("Tool graph (the output of the tool on the left can be an input of the tools on the right; the type "
                "passed in brackets):" if d["dep"] == "resource" else
                "Tool graph (the tool on the left can be followed by the tools on the right):")
        msg += "\n\n" + head + "\n" + ("\n".join(gl) if gl else "(no edges among these tools)")
    return msg + f"\n\nUser request: {d['samples'][i]['user_request']}"

METHODS = ("all", "embed", "graph")
for dom, d in DATA.items():
    d["plans"] = {}
    for m in METHODS:
        reqs = {f"{m}|{r['id']}": LLM.body([{"role": "system", "content": sys_prompt(d)},
                                            {"role": "user", "content": prompt(d, i, m)}], max_tokens=600, schema=Plan)
                for i, r in enumerate(d["samples"])}
        d["plans"][m] = llm.run(f"nb26_tb_{dom}_{m}", reqs, parse=Plan)
    bad = {m: sum(v["parsed"] is None for v in d["plans"][m].values()) for m in METHODS}
    print(f"{dom}: {len(d['samples']) * len(METHODS):,} plans; unparsed per condition {bad}")

d = DATA["multimedia"]
print("\nExample request:", d["samples"][0]["user_request"])
print("Gold nodes:", sorted(gold_nodes(d["samples"][0])), "| gold links:", sorted(gold_links(d["samples"][0])))
print("\n(c) prompt:\n" + prompt(d, 0, "graph"))

huggingface: 1,500 plans; unparsed per condition {'all': 0, 'embed': 0, 'graph': 0}


multimedia: 1,500 plans; unparsed per condition {'all': 0, 'embed': 0, 'graph': 0}


[nb26_tb_dailylifeapis_graph] 500 new requests -> batch (0 cached)


[nb26_tb_dailylifeapis_graph] resuming: 1 running, 0 queued


[nb26_tb_dailylifeapis_graph] batch batch_6ae1974d5e2c8190b893bfd016e476d9 completed: 500 ok, 0 failed


[nb26_tb_dailylifeapis_graph] 1 min: 0 running, 0 queued, 500 results cached


[nb26_tb_dailylifeapis_graph] done in 0.5 min


dailylifeapis: 1,500 plans; unparsed per condition {'all': 0, 'embed': 0, 'graph': 0}

Example request: I have a non-English text: 'Bonjour, voici le lien de la vidéo incroyable: https://www.example.com/example.mp4. J'aimerais avoir cette vidéo en anglais et également un fichier audio de la description en anglais.'
Gold nodes: ['Text Translator', 'Text-to-Audio', 'URL Extractor', 'Video Downloader'] | gold links: [('Text Translator', 'Text-to-Audio'), ('Text Translator', 'URL Extractor'), ('URL Extractor', 'Video Downloader')]

(c) prompt:
Tools (name: description, input and output types):
- Text-to-Video: Generates a video based on a given text description. Input types: text. Output types: video.
- Text-to-Audio: Generates an audio file based on a given text description. Input types: text. Output types: audio.
- Video Downloader: Downloads a video from a given URL. Input types: url. Output types: video.
- Video-to-Text: Transcribes speech from a video file into text. Input types: vide

## Scores per sample

Predicted tool names are matched to the tool list case-insensitively; a name outside the list is a false positive.
**Node-F1** compares the set of predicted tools with the set of gold tools. **Edge-F1** compares the predicted links
(both ends mapped to tool names) with the gold links, on samples with at least one gold link. **Exact** = the node
sets are equal. TaskBench's own figures are micro-averaged over all samples (`micro`).

In [5]:
def f1(pred, gold):
    if not pred and not gold:
        return 1.0
    tp = len(pred & gold)
    return 2 * tp / (len(pred) + len(gold)) if (pred or gold) else 0.0

def score(d, i, m):
    r = d["samples"][i]
    res = d["plans"][m][f"{m}|{r['id']}"]
    p = res["parsed"] or {"nodes": [], "links": []}
    c = lambda x: d["canon"].get(str(x).lower().strip(), str(x))
    pn = {c(x) for x in p["nodes"]}
    pl = {(c(l["source"]), c(l["target"])) for l in p["links"]}
    gn, gl = gold_nodes(r), gold_links(r)
    return {"node_f1": f1(pn, gn), "edge_f1": f1(pl, gl) if gl else float("nan"), "exact": float(pn == gn),
            "cand_rec": len(gn & set(d["cand"][m][i])) / len(gn), "tok_in": res["in"], "tok_out": res["out"],
            "tp": len(pn & gn), "fp": len(pn - gn), "fn": len(gn - pn),
            "ltp": len(pl & gl), "lfp": len(pl - gl), "lfn": len(gl - pl), "has_links": bool(gl)}

for dom, d in DATA.items():
    d["scores"] = [{"domain": dom, "type": r["type"], **{m: score(d, i, m) for m in METHODS}}
                   for i, r in enumerate(d["samples"])]
ALL = [s for d in DATA.values() for s in d["scores"]]
print("scored", len(ALL), "samples x", len(METHODS), "conditions")

scored 1500 samples x 3 conditions


## Results with 95% confidence intervals

Means per condition and paired bootstrap intervals over samples. **Primary: node-F1, graph − embed, all domains.**

In [6]:
METRICS = [("node_f1", "node-F1 (primary)"), ("edge_f1", "edge-F1"), ("exact", "exact node set"),
           ("cand_rec", "candidate recall"), ("tok_in", "prompt tokens")]
PAIRS = [("embed", "graph"), ("all", "graph"), ("all", "embed")]
SUMMARY = {}

def table(label, rows):
    print(f"\n{label}  (n = {len(rows)})")
    print(f"{'metric':<20}{'all':>8}{'embed':>8}{'graph':>8}" + "".join(f"{f'{b} - {a} [95% CI]':>28}" for a, b in PAIRS))
    for k, lab in METRICS:
        ok = [r for r in rows if not np.isnan(r["all"][k])]
        if not ok:
            continue
        means = {m: np.mean([r[m][k] for r in ok]) for m in METHODS}
        diffs = {(a, b): paired_boot([r[a][k] for r in ok], [r[b][k] for r in ok]) for a, b in PAIRS}
        SUMMARY[(label, k)] = (len(ok), means, diffs)
        if k == "tok_in":
            fm = lambda x: f"{x:,.0f}"; fc = lambda t: f"{t[0]:+,.0f} [{t[1]:+,.0f}, {t[2]:+,.0f}]"
        else:
            fm = lambda x: f"{x:.1%}"; fc = lambda t: f"{t[0]:+.1%} [{t[1]:+.1%}, {t[2]:+.1%}]"
        print(f"{lab:<20}" + "".join(f"{fm(means[m]):>8}" for m in METHODS) + "".join(f"{fc(diffs[p]):>28}" for p in PAIRS))

for dom, d in DATA.items():
    table(dom, d["scores"])
table("all domains", ALL)


huggingface  (n = 500)
metric                   all   embed   graph      graph - embed [95% CI]        graph - all [95% CI]        embed - all [95% CI]
node-F1 (primary)      87.0%   81.5%   79.7%        -1.8% [-3.2%, -0.5%]        -7.3% [-9.2%, -5.3%]        -5.5% [-7.2%, -3.7%]


edge-F1                50.0%   37.0%   38.1%        +1.1% [-2.3%, +4.6%]      -11.9% [-16.2%, -7.5%]      -13.0% [-16.9%, -9.0%]


exact node set         62.6%   51.6%   48.0%        -3.6% [-6.0%, -1.2%]     -14.6% [-18.4%, -11.0%]      -11.0% [-14.4%, -7.6%]


candidate recall      100.0%   85.6%   83.1%        -2.5% [-3.6%, -1.4%]     -16.9% [-18.9%, -14.9%]     -14.4% [-16.2%, -12.5%]


prompt tokens          1,439     679     978           +298 [+295, +302]           -462 [-465, -458]           -760 [-763, -757]

multimedia  (n = 500)
metric                   all   embed   graph      graph - embed [95% CI]        graph - all [95% CI]        embed - all [95% CI]
node-F1 (primary)      91.8%   87.0%   86.0%        -1.0% [-2.4%, +0.4%]        -5.8% [-7.9%, -3.8%]        -4.9% [-6.8%, -2.9%]
edge-F1                68.8%   54.5%   54.3%        -0.2% [-3.4%, +3.0%]     -14.5% [-18.8%, -10.0%]     -14.3% [-18.5%, -10.3%]
exact node set         72.6%   60.0%   58.8%        -1.2% [-3.8%, +1.4%]      -13.8% [-18.0%, -9.6%]      -12.6% [-16.8%, -8.4%]
candidate recall      100.0%   86.7%   85.5%        -1.2% [-2.4%, -0.0%]     -14.5% [-16.4%, -12.7%]     -13.3% [-15.1%, -11.6%]


prompt tokens          1,442     515     800           +285 [+281, +289]           -642 [-646, -637]           -927 [-928, -925]

dailylifeapis  (n = 500)
metric                   all   embed   graph      graph - embed [95% CI]        graph - all [95% CI]        embed - all [95% CI]


... (3 outputs left out)

node-F1 (primary)      91.9%   86.1%   85.7%        -0.4% [-1.1%, +0.4%]        -6.2% [-7.3%, -5.2%]        -5.8% [-6.8%, -4.9%]


edge-F1                55.4%   43.3%   53.0%       +9.7% [+7.5%, +11.8%]        -2.4% [-5.2%, +0.4%]      -12.1% [-14.3%, -9.8%]
exact node set         74.6%   58.1%   57.1%        -1.0% [-2.3%, +0.3%]     -17.5% [-19.9%, -15.2%]     -16.5% [-18.7%, -14.2%]


candidate recall      100.0%   87.3%   86.1%        -1.2% [-1.8%, -0.7%]     -13.9% [-15.0%, -12.9%]     -12.7% [-13.6%, -11.7%]


prompt tokens          1,347     554     845           +292 [+290, +293]           -501 [-507, -496]           -793 [-798, -788]


In [7]:
print("Node-F1 by structure of the gold graph (all domains)")
for t in ("single", "chain", "dag"):
    rows = [s for s in ALL if s["type"] == t]
    if rows:
        mm = {m: np.mean([r[m]["node_f1"] for r in rows]) for m in METHODS}
        g = paired_boot([r["embed"]["node_f1"] for r in rows], [r["graph"]["node_f1"] for r in rows])
        print(f"  {t:<7} n={len(rows):<4} all {mm['all']:.1%}  embed {mm['embed']:.1%}  graph {mm['graph']:.1%}  "
              f"graph - embed {g[0]:+.1%} [{g[1]:+.1%}, {g[2]:+.1%}]")

print("\nTaskBench-style micro F1 (pooled counts, as in TaskBench's evaluate.py)")
for dom in list(DATA) + ["all domains"]:
    rows = ALL if dom == "all domains" else DATA[dom]["scores"]
    out = []
    for m in METHODS:
        tp, fp, fn = (sum(r[m][x] for r in rows) for x in ("tp", "fp", "fn"))
        lr = [r for r in rows if r[m]["has_links"]]
        ltp, lfp, lfn = (sum(r[m][x] for r in lr) for x in ("ltp", "lfp", "lfn"))
        out.append(f"{m} node {2 * tp / max(2 * tp + fp + fn, 1):.1%} / link {2 * ltp / max(2 * ltp + lfp + lfn, 1):.1%}")
    print(f"  {dom:<14} " + "  |  ".join(out))

Node-F1 by structure of the gold graph (all domains)
  single  n=522  all 96.8%  embed 98.4%  graph 98.7%  graph - embed +0.3% [-0.4%, +1.1%]
  chain   n=843  all 89.3%  embed 79.7%  graph 79.3%  graph - embed -0.4% [-1.5%, +0.7%]
  dag     n=135  all 89.4%  embed 78.5%  graph 75.4%  graph - embed -3.1% [-6.0%, -0.4%]

TaskBench-style micro F1 (pooled counts, as in TaskBench's evaluate.py)
  huggingface    all node 84.2% / link 53.5%  |  embed node 76.1% / link 39.7%  |  graph node 73.3% / link 39.4%
  multimedia     all node 90.4% / link 69.8%  |  embed node 82.1% / link 53.9%  |  graph node 80.5% / link 52.0%
  dailylifeapis  all node 97.1% / link 57.3%  |  embed node 87.4% / link 43.9%  |  graph node 88.3% / link 63.4%
  all domains    all node 90.9% / link 60.6%  |  embed node 82.1% / link 46.1%  |  graph node 81.1% / link 52.7%


### One request, three plans

The first sample on which (c) and (b) differ in node-F1 (an illustration, not a typical case).

In [8]:
hit = next(((dom, i) for dom, d in DATA.items() for i, s in enumerate(d["scores"])
            if s["graph"]["node_f1"] != s["embed"]["node_f1"]), (DOMAINS[0], 0))
dom, i = hit; d = DATA[dom]; r = d["samples"][i]
print(f"[{dom}] {r['user_request']}\nGold nodes: {sorted(gold_nodes(r))}\nGold links: {sorted(gold_links(r))}")
for m in METHODS:
    p = d["plans"][m][f"{m}|{r['id']}"]["parsed"] or {}
    print(f"\n{m.upper()} (node-F1 {d['scores'][i][m]['node_f1']:.0%}); candidates: "
          f"{'all ' + str(len(d['names'])) if m == 'all' else d['cand'][m][i]}")
    print("   nodes:", p.get("nodes"), "\n   links:", [(l['source'], l['target']) for l in p.get("links", [])])

[huggingface] I am attaching an example.wav audio file. I need to enhance the audio quality, identify the speaker's emotion, and generate an appropriate conversational response.
Gold nodes: ['Audio Classification', 'Audio-to-Audio', 'Conversational']
Gold links: [('Audio Classification', 'Conversational'), ('Audio-to-Audio', 'Audio Classification')]

ALL (node-F1 86%); candidates: all 23
   nodes: ['Audio-to-Audio', 'Audio Classification', 'Automatic Speech Recognition', 'Conversational'] 
   links: [('Automatic Speech Recognition', 'Conversational')]

EMBED (node-F1 86%); candidates: ['Audio Classification', 'Audio-to-Audio', 'Conversational', 'Text-to-Speech', 'Document Question Answering', 'Automatic Speech Recognition', 'Text-to-Video', 'Visual Question Answering']
   nodes: ['Audio-to-Audio', 'Audio Classification', 'Automatic Speech Recognition', 'Conversational'] 
   links: [('Automatic Speech Recognition', 'Conversational')]

GRAPH (node-F1 100%); candidates: ['Audio Classifica

# Part 2: ToolBench, retrieval from ~47k APIs

## Data: test queries and the API corpus (downloaded once, by range requests)

The official ToolBench download links are dead, so the official `data.zip` (1.76 GB) is read from a public
re-upload on Hugging Face at a pinned revision. `RangeFile` lets `zipfile` read only the members it needs (the six
test-query files and the 10,657 tool JSONs, about 170 MB) over HTTP range requests. Each API is reduced to the fields
used here and written to `corpus.jsonl`; the queries to `queries.json`. No code from the archive is used.

In [9]:
import io, time, zipfile

TB = "data/benchmarks/toolbench"
TB_URL = ("https://huggingface.co/datasets/nullwwg/toolbench-data/resolve/"
          "f31e7988f2ddf5d2d1633c6614cce45834d24f86/data.zip")          # re-upload of ToolBench's official data.zip
TEST_SETS = ("G1_instruction", "G1_category", "G1_tool", "G2_instruction", "G2_category", "G3_instruction")

class RangeFile(io.RawIOBase):
    # Seekable read-only view of a remote file over HTTP range requests, cached in 8 MB blocks.
    BLOCK = 8 << 20
    def __init__(self, url):
        r = urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=60)
        self.url, self.size, self.pos, self.blocks = r.url, int(r.headers["Content-Length"]), 0, {}
    def readable(self): return True
    def seekable(self): return True
    def tell(self): return self.pos
    def seek(self, off, whence=0):
        self.pos = off if whence == 0 else self.pos + off if whence == 1 else self.size + off
        return self.pos
    def _block(self, b):
        if b not in self.blocks:
            if len(self.blocks) > 8:
                self.blocks.pop(next(iter(self.blocks)))
            lo, hi = b * self.BLOCK, min(self.size, (b + 1) * self.BLOCK) - 1
            for attempt in range(8):
                try:
                    req = urllib.request.Request(self.url, headers={"Range": f"bytes={lo}-{hi}"})
                    self.blocks[b] = urllib.request.urlopen(req, timeout=120).read(); break
                except Exception:
                    if attempt == 7:
                        raise
                    time.sleep(2 ** attempt)
        return self.blocks[b]
    def readinto(self, buf):
        if self.pos >= self.size:
            return 0
        b, o = divmod(self.pos, self.BLOCK)
        data = self._block(b)[o:o + len(buf)]
        buf[:len(data)] = data; self.pos += len(data)
        return len(data)

def params(ps):
    return [f"{p.get('name')} ({str(p.get('type', '')).lower()})" for p in (ps or []) if isinstance(p, dict)]

if not (os.path.exists(f"{TB}/corpus.jsonl") and os.path.exists(f"{TB}/queries.json")):
    os.makedirs(TB, exist_ok=True)
    z = zipfile.ZipFile(io.BufferedReader(RangeFile(TB_URL), 1 << 16))
    infos = sorted(z.infolist(), key=lambda i: i.header_offset)
    queries = []
    for i in infos:
        name = i.filename.rsplit("/", 1)[-1][:-5]
        if i.filename.startswith("data/test_instruction/") and name in TEST_SETS:
            for q in json.loads(z.read(i)):
                queries.append({"qid": f"{name}|{q['query_id']}", "group": name, "query": q["query"],
                                "gold": [list(x) for x in q["relevant APIs"]]})
    seen, n_raw = set(), 0
    with open(f"{TB}/corpus.jsonl", "w") as f:
        for i in infos:
            parts = i.filename.split("/")
            if not (i.filename.startswith("data/toolenv/tools/") and len(parts) == 5 and parts[4].endswith(".json")):
                continue
            t = json.loads(z.read(i))
            for a in t.get("api_list", []):
                n_raw += 1
                key = (t.get("tool_name"), a.get("name"))
                if key in seen:
                    continue
                seen.add(key)
                f.write(json.dumps({"category": parts[3], "tool": t.get("tool_name"),
                                    "tool_desc": (t.get("tool_description") or "")[:200], "api": a.get("name"),
                                    "desc": (a.get("description") or "")[:300],
                                    "params": params(a.get("required_parameters")) + params(a.get("optional_parameters"))
                                    }) + "\n")
    json.dump(queries, open(f"{TB}/queries.json", "w"))
    print(f"extracted {len(queries):,} test queries and {len(seen):,} APIs ({n_raw:,} API entries before dedup)")

CORPUS = [json.loads(l) for l in open(f"{TB}/corpus.jsonl")]
API_ID = {(a["tool"], a["api"]): j for j, a in enumerate(CORPUS)}
TOOL_APIS = defaultdict(list)                       # the hierarchy: tool -> its APIs (category is a field of each API)
for j, a in enumerate(CORPUS):
    TOOL_APIS[(a["category"], a["tool"])].append(j)
TB_ALL = json.load(open(f"{TB}/queries.json"))
TB_N = 1000
TB_Q = random.Random(7).sample(TB_ALL, TB_N)[:25 if TEST else TB_N]
for q in TB_Q:
    q["gold_ids"] = {API_ID[tuple(g)] for g in q["gold"] if tuple(g) in API_ID}
n_gold = sum(len({tuple(g) for g in q["gold"]}) for q in TB_Q)
print(f"corpus: {len(CORPUS):,} APIs, {len(TOOL_APIS):,} tools, {len({a['category'] for a in CORPUS})} categories")
print(f"queries: {len(TB_ALL):,} in the six test sets, {len(TB_Q)} used | groups {dict(Counter(q['group'] for q in TB_Q))}")
print(f"gold APIs per query {np.mean([len({tuple(g) for g in q['gold']}) for q in TB_Q]):.2f}; "
      f"gold APIs found in the corpus {sum(len(q['gold_ids']) for q in TB_Q) / n_gold:.1%}")

corpus: 46,985 APIs, 10,601 tools, 50 categories
queries: 1,100 in the six test sets, 1000 used | groups {'G1_tool': 185, 'G2_category': 182, 'G2_instruction': 183, 'G1_category': 179, 'G3_instruction': 89, 'G1_instruction': 182}
gold APIs per query 2.40; gold APIs found in the corpus 99.0%


## Candidates: (b) embedding top-50 and (c) the category → tool → API hierarchy

The corpus embeddings are cached in `cache_embed/`, shared by test and full runs (the corpus is the same).

In [10]:
def api_text(a):
    ps = ", ".join(a["params"]) or "none"
    return (f"{a['category']} > {a['tool']}: {a['tool_desc']}\nAPI {a['api']}: {a['desc']}\nParameters: {ps}")

emb = LLM(f"{TB}/cache_embed", mode=MODE)
TB_P = emb.embed("nb26_toolbench_corpus", [api_text(a) for a in CORPUS], cap=1000)
TB_QV = emb.embed(f"nb26_toolbench_queries_{len(TB_Q)}", [q["query"] for q in TB_Q])

N_CAND, TB_SEEDS, PER_TOOL = 50, 10, 8

def cands_tb(qi):
    sim = TB_P @ TB_QV[qi]
    top = np.argsort(-sim)[:1000]
    embed = [int(j) for j in top[:N_CAND]]
    tools = list(dict.fromkeys((CORPUS[j]["category"], CORPUS[j]["tool"]) for j in top[:TB_SEEDS]))
    graph = []
    for t in tools:                                 # seed APIs -> their tools -> the tools' APIs, by similarity
        graph += sorted(TOOL_APIS[t], key=lambda j: -sim[j])[:PER_TOOL]
    graph = list(dict.fromkeys(graph))[:N_CAND]
    for j in top:                                   # fill to 50 with the embedding order
        if len(graph) >= N_CAND:
            break
        if int(j) not in graph:
            graph.append(int(j))
    return {"embed": embed, "graph": graph}

def recall_at(ranked, gold, k):
    return len(set(ranked[:k]) & gold) / len(gold) if gold else 0.0

def ndcg_at(ranked, gold, k):
    dcg = sum(1 / np.log2(r + 2) for r, j in enumerate(ranked[:k]) if j in gold)
    idcg = sum(1 / np.log2(r + 2) for r in range(min(k, len(gold))))
    return dcg / idcg if idcg else 0.0

def n_gold_q(q):                                   # gold APIs incl. the ~1% not found in the corpus (unreachable)
    return len({tuple(g) for g in q["gold"]})

TB_C = [cands_tb(i) for i in range(len(TB_Q))]
for m in ("embed", "graph"):
    r50 = np.mean([len(set(c[m]) & q["gold_ids"]) / n_gold_q(q) for q, c in zip(TB_Q, TB_C)])
    r5 = np.mean([len(set(c[m][:5]) & q["gold_ids"]) / n_gold_q(q) for q, c in zip(TB_Q, TB_C)])
    print(f"{m:<6} candidate recall@50 {r50:.1%} | recall@5 of the candidate order (no rerank) {r5:.1%} | "
          f"tools per candidate list {np.mean([len({CORPUS[j]['tool'] for j in c[m]}) for c in TB_C]):.1f}")

embed  candidate recall@50 70.0% | recall@5 of the candidate order (no rerank) 45.2% | tools per candidate list 23.8
graph  candidate recall@50 70.6% | recall@5 of the candidate order (no rerank) 44.7% | tools per candidate list 20.4


## The reranker (one Batch job per condition)

The same system prompt and the same formatting for both candidate lists: grouped by tool in the order the tools first
appear, numbered 1–50. The LLM returns the numbers of the APIs the query needs, most relevant first (at most 10).

In [11]:
class Ranked(BaseModel):
    ranked: list[int] = Field(description="numbers of the APIs needed for the query, most relevant first (at most 10)")

RERANK_SYS = ("You select APIs for a user query. You receive the query and a numbered list of candidate APIs, grouped "
              "by tool. Return the numbers of the APIs that are needed to fulfil the query, the most relevant first, "
              "at most 10. Include every API the query needs; leave out APIs it does not need.")

def rerank_prompt(qi, cands):
    groups = defaultdict(list)
    for n, j in enumerate(cands, 1):
        groups[(CORPUS[j]["category"], CORPUS[j]["tool"])].append((n, j))
    lines = []
    for (cat, tool), items in groups.items():
        lines.append(f"Tool: {tool} (category {cat}) - {CORPUS[items[0][1]]['tool_desc']}")
        for n, j in items:
            a = CORPUS[j]
            lines.append(f"  [{n}] {a['api']}: {a['desc'][:200]} | parameters: {', '.join(a['params'][:8]) or 'none'}")
    return "Candidate APIs:\n" + "\n".join(lines) + f"\n\nQuery: {TB_Q[qi]['query']}"

TB_RES = {}
for m in ("embed", "graph"):
    reqs = {f"{m}|{q['qid']}": LLM.body([{"role": "system", "content": RERANK_SYS},
                                         {"role": "user", "content": rerank_prompt(i, TB_C[i][m])}],
                                        max_tokens=150, schema=Ranked)
            for i, q in enumerate(TB_Q)}
    TB_RES[m] = llm.run(f"nb26_toolbench_rerank_{m}", reqs, parse=Ranked)
    print(f"{m}: {len(reqs)} rerank calls, {sum(r['parsed'] is None for r in TB_RES[m].values())} unparsed")

def final_rank(qi, m):
    cands = TB_C[qi][m]
    p = TB_RES[m][f"{m}|{TB_Q[qi]['qid']}"]["parsed"] or {"ranked": []}
    chosen = [cands[n - 1] for n in p["ranked"] if isinstance(n, int) and 1 <= n <= len(cands)]
    chosen = list(dict.fromkeys(chosen))
    return chosen + [j for j in cands if j not in chosen], len(chosen)

TB_S = []
for i, q in enumerate(TB_Q):
    row = {"group": q["group"]}
    for m in ("embed", "graph"):
        rk, n_sel = final_rank(i, m)
        g, ng = q["gold_ids"], n_gold_q(q)
        row[m] = {"r5": len(set(rk[:5]) & g) / ng, "ndcg5": ndcg_at(rk, g, 5), "r10": len(set(rk[:10]) & g) / ng,
                  "r50": len(set(TB_C[i][m]) & g) / ng, "r5_norerank": len(set(TB_C[i][m][:5]) & g) / ng,
                  "n_sel": n_sel, "tok_in": TB_RES[m][f"{m}|{q['qid']}"]["in"]}
    TB_S.append(row)
print(f"scored {len(TB_S)} queries")

[nb26_toolbench_rerank_embed] 1,000 new requests -> batch (0 cached)


[nb26_toolbench_rerank_embed] uploaded input file 1/1 (1,000 requests, 13 MB)


[nb26_toolbench_rerank_embed] started batch batch_6aea9a0908c88190ab2abb8577118e40


[nb26_toolbench_rerank_embed] 1 min: 1 running, 0 queued, 0 results cached


[nb26_toolbench_rerank_embed] 6 min: 1 running, 0 queued, 0 results cached


[nb26_toolbench_rerank_embed] 11 min: 1 running, 0 queued, 0 results cached


... (59 outputs left out)

[nb26_toolbench_rerank_graph] 96 min: 1 running, 0 queued, 0 results cached


[nb26_toolbench_rerank_graph] batch batch_6aeac69253b88190ab28efe0dbb50910 completed: 1,000 ok, 0 failed


[nb26_toolbench_rerank_graph] done in 101.1 min


graph: 1000 rerank calls, 0 unparsed
scored 1000 queries


## ToolBench results with 95% confidence intervals

**Primary: recall@5 after the rerank, graph − embedding.**

In [12]:
TB_METRICS = [("r5", "recall@5 (primary)"), ("ndcg5", "NDCG@5"), ("r10", "recall@10"),
              ("r50", "candidate recall@50"), ("r5_norerank", "recall@5, no rerank"), ("n_sel", "APIs selected"),
              ("tok_in", "prompt tokens")]
TB_SUM = {}

def tb_table(label, rows):
    print(f"\n{label}  (n = {len(rows)})")
    print(f"{'metric':<24}{'embed':>9}{'graph':>9}{'graph - embed [95% CI]':>30}")
    for k, lab in TB_METRICS:
        a, b = [r["embed"][k] for r in rows], [r["graph"][k] for r in rows]
        d = paired_boot(a, b)
        TB_SUM[(label, k)] = (len(rows), np.mean(a), np.mean(b), d)
        if k in ("tok_in", "n_sel"):
            print(f"{lab:<24}{np.mean(a):>9,.1f}{np.mean(b):>9,.1f}{f'{d[0]:+,.1f} [{d[1]:+,.1f}, {d[2]:+,.1f}]':>30}")
        else:
            print(f"{lab:<24}{np.mean(a):>9.1%}{np.mean(b):>9.1%}{f'{d[0]:+.1%} [{d[1]:+.1%}, {d[2]:+.1%}]':>30}")

tb_table("ToolBench, all test sets", TB_S)
print("\nRecall@5 by test set")
for g in TEST_SETS:
    rows = [r for r in TB_S if r["group"] == g]
    if rows:
        d = paired_boot([r["embed"]["r5"] for r in rows], [r["graph"]["r5"] for r in rows])
        print(f"  {g:<16} n={len(rows):<4} embed {np.mean([r['embed']['r5'] for r in rows]):.1%}  "
              f"graph {np.mean([r['graph']['r5'] for r in rows]):.1%}  diff {d[0]:+.1%} [{d[1]:+.1%}, {d[2]:+.1%}]")


ToolBench, all test sets  (n = 1000)
metric                      embed    graph        graph - embed [95% CI]
recall@5 (primary)          57.1%    57.9%          +0.8% [-0.3%, +2.0%]
NDCG@5                      55.7%    56.8%          +1.1% [+0.1%, +2.1%]
recall@10                   61.0%    61.0%          +0.0% [-1.0%, +1.1%]
candidate recall@50         70.0%    70.6%          +0.6% [-0.3%, +1.5%]
recall@5, no rerank         45.2%    44.7%          -0.5% [-1.8%, +0.9%]
APIs selected                 2.7      2.7             -0.0 [-0.1, +0.1]
prompt tokens             2,893.2  2,782.6        -110.6 [-122.8, -98.6]

Recall@5 by test set
  G1_instruction   n=182  embed 70.2%  graph 70.9%  diff +0.7% [-1.7%, +3.2%]
  G1_category      n=179  embed 73.4%  graph 76.0%  diff +2.6% [-0.6%, +5.9%]
  G1_tool          n=185  embed 66.0%  graph 67.7%  diff +1.7% [-0.8%, +4.3%]
  G2_instruction   n=183  embed 47.2%  graph 45.7%  diff -1.5% [-3.9%, +1.0%]
  G2_category      n=182  embed 41.3%  graph

In [13]:
# One query: the gold APIs and the top 5 of each condition (* = gold)
i = next((i for i, r in enumerate(TB_S) if r["graph"]["r5"] != r["embed"]["r5"]), 0)
q = TB_Q[i]
print("Query:", q["query"]); print("Gold:", q["gold"])
for m in ("embed", "graph"):
    rk, _ = final_rank(i, m)
    print(f"\n{m.upper()} (recall@5 {TB_S[i][m]['r5']:.0%}):")
    for j in rk[:5]:
        print(f"  {'*' if j in q['gold_ids'] else ' '} {CORPUS[j]['tool']} / {CORPUS[j]['api']}")

Query: I'm organizing a movie marathon for my family and we want to watch movies from various genres. Can you suggest a comedy movie, an action movie, and a romance movie from IMDb's top 100 list? Additionally, provide us with the streaming availability information for these movies on popular platforms like Netflix, Prime Video, and Hulu.
Gold: [['IMDb top 100 movies', 'Get Top 100 Movies according to IMDb'], ['Streaming Availability', 'Search Ultra']]

EMBED (recall@5 50%):
    IMDb Top 100 Movies / Top 100 movies list
    Streaming Availability / Get Basic (FREE)
    TOP 10 movies of the day / Get all the best movies of the day
    Streaming Availability / Genres (FREE)
  * IMDb top 100 movies / Get Top 100 Movies according to IMDb

GRAPH (recall@5 0%):
    IMDb Top 100 Movies / Top 100 movies list
    IMDb Top 100 Movies / Movie Data By Id
    Watchmode / Title Streaming Sources
    TOP 10 movies of the day / Get all the best movies of the day
    TOP 10 movies of the day / Get top 

## What this test cost

In [14]:
STAGES = [f"nb26_tb_{dom}_{m}" for dom in DATA for m in METHODS]
TB_STAGES = ["nb26_toolbench_rerank_embed", "nb26_toolbench_rerank_graph"]
LEDGER = llm.ledger(STAGES + TB_STAGES)
LEDGER["(embeddings, TaskBench)"] = llm.ledger([])["(embeddings)"]
LEDGER["(embeddings, ToolBench)"] = emb.ledger([])["(embeddings)"]
del LEDGER["(embeddings)"]
print(f"{'stage':<34}{'calls':>8}{'input tok':>13}{'output tok':>12}{'USD':>8}{'via Batch':>11}")
for st, a in LEDGER.items():
    print(f"{st:<34}{a['calls']:>8,}{a['in']:>13,}{a['out']:>12,}{a['usd']:>8.3f}{a['batch_share']:>11.0%}")
tot = sum(a["usd"] for a in LEDGER.values())
print(f"{'TOTAL':<34}{sum(a['calls'] for a in LEDGER.values()):>8,}{'':>25}{tot:>8.3f}")
if TEST:
    # LLM calls scale with n and halve in Batch; the ToolBench corpus embedding is paid once and shared (cache_embed).
    p1 = sum(LEDGER[st]["usd"] for st in STAGES) * (N_PER / N_RUN) / 2
    p2 = sum(LEDGER[st]["usd"] for st in TB_STAGES) * (TB_N / len(TB_Q)) / 2
    print(f"\nForecast of the full run (Batch): TaskBench {len(STAGES) * N_PER:,} calls ~${p1:.2f} | ToolBench "
          f"{2 * TB_N:,} calls ~${p2:.2f} (+ query embeddings, < $0.01; corpus embeddings already cached)")

stage                                calls    input tok  output tok     USD  via Batch
nb26_tb_huggingface_all                500      719,686      19,684   0.041       100%
nb26_tb_huggingface_embed              500      339,610      17,307   0.021       100%
nb26_tb_huggingface_graph              500      488,841      18,253   0.029       100%
nb26_tb_multimedia_all                 500      720,899      23,561   0.042       100%
nb26_tb_multimedia_embed               500      257,565      19,709   0.018       100%
nb26_tb_multimedia_graph               500      400,040      20,225   0.025       100%
nb26_tb_dailylifeapis_all              500      579,229      19,855   0.034       100%
nb26_tb_dailylifeapis_embed            500      233,378      17,419   0.016       100%
nb26_tb_dailylifeapis_graph            500      379,018      22,189   0.024       100%
nb26_toolbench_rerank_embed          1,000    2,893,236      16,464   0.149       100%
nb26_toolbench_rerank_graph          1,000 

## Limitations

- **Upper bound for the graph, in both parts.** TaskBench requests were written from subgraphs of the same tool graph,
  so the gold links are always graph edges. ToolBench queries were written from APIs sampled within one tool (G1), one
  category (G2) or one collection (G3), the hierarchy the graph condition expands along. A tool graph built
  independently of the tasks would help less.
- **Small, dense tool sets in Part 1.** 23–40 tools fit in the prompt, and the graphs are dense (daily-life complete),
  so the graph mostly re-ranks and annotates the candidates instead of pruning them. Part 2 is the large-tool-set case.
- **Tool names are public task names** in TaskBench (Hugging Face task categories), which the model may know without
  the list; a correct tool outside the candidates still counts.
- **No parameters, no execution.** Only tool choice, links (Part 1) and retrieval ranks (Part 2) are scored, not
  argument values or task success.
- **Part 2 data source.** A third-party re-upload of ToolBench's official archive (the official links are dead); the
  corpus is ToolBench's full tool environment (~47k APIs), not the 16,464-API subset the paper reports.
- **One configuration.** One model, one prompt per part, k = 8 with 3 seeds (Part 1), 50 candidates with 10 seeds and
  8 APIs per tool (Part 2); each configuration run once. The embedding model is not a fine-tuned tool retriever.

**References.** Shen et al., *TaskBench: Benchmarking Large Language Models for Task Automation*, NeurIPS 2024 D&B ·
Qin et al., *ToolLLM: Facilitating Large Language Models to Master 16000+ Real-world APIs* (ToolBench), ICLR 2024 ·
Wu et al., *Can Graph Learning Improve Planning in LLM-based Agents?*, NeurIPS 2024 · Liu et al., *ControlLLM*,
arXiv 2310.17796 · Liu et al., *ToolNet*, arXiv 2403.00839 · Gao et al., *Tool Graph Retriever*, arXiv 2508.05152 ·
Shi et al., *ToolRet*, arXiv 2503.01763 (not used: no data licence).

In [15]:
print("=" * 120)
print(f"RESULTS SUMMARY — Notebook 4.2: tool selection over a tool graph, {MODE} run, gpt-6-luna (no reasoning)")
print("=" * 120)
print("Part 1, TaskBench (planner over 23-40 tools; graph = published type-compatible tool graph)")
print(f"{'domain':<15}{'metric':<20}{'all':>8}{'embed':>8}{'graph':>8}{'graph - embed [95% CI]':>28}{'graph - all [95% CI]':>28}")
for label in list(DATA) + ["all domains"]:
    for k, lab in METRICS:
        if (label, k) not in SUMMARY or k == "cand_rec":
            continue
        n, means, diffs = SUMMARY[(label, k)]
        if k == "tok_in":
            fm = lambda x: f"{x:,.0f}"; fc = lambda t: f"{t[0]:+,.0f} [{t[1]:+,.0f}, {t[2]:+,.0f}]"
        else:
            fm = lambda x: f"{x:.1%}"; fc = lambda t: f"{t[0]:+.1%} [{t[1]:+.1%}, {t[2]:+.1%}]"
        print(f"{label:<15}{lab:<20}" + "".join(f"{fm(means[m]):>8}" for m in METHODS) +
              f"{fc(diffs[('embed', 'graph')]):>28}{fc(diffs[('all', 'graph')]):>28}")
verdict = lambda g: "better" if g[1] > 0 else "worse" if g[2] < 0 else "no resolvable difference"
n, means, diffs = SUMMARY[("all domains", "node_f1")]
g = diffs[("embed", "graph")]
print(f"\nPart 1 primary (node-F1, graph - embedding top-{K}, all domains, n = {n}): {g[0]:+.1%} [{g[1]:+.1%}, {g[2]:+.1%}]"
      f" -> {verdict(g)}; half-width {(g[2] - g[1]) / 2:.1%}")

print(f"\nPart 2, ToolBench (rerank of 50 candidates from {len(CORPUS):,} APIs; graph = category -> tool -> API hierarchy)")
print(f"{'metric':<24}{'embed':>9}{'graph':>9}{'graph - embed [95% CI]':>30}")
for k, lab in TB_METRICS:
    n2, a, b, d = TB_SUM[("ToolBench, all test sets", k)]
    if k in ("tok_in", "n_sel"):
        print(f"{lab:<24}{a:>9,.1f}{b:>9,.1f}{f'{d[0]:+,.1f} [{d[1]:+,.1f}, {d[2]:+,.1f}]':>30}")
    else:
        print(f"{lab:<24}{a:>9.1%}{b:>9.1%}{f'{d[0]:+.1%} [{d[1]:+.1%}, {d[2]:+.1%}]':>30}")
n2, a, b, d = TB_SUM[("ToolBench, all test sets", "r5")]
print(f"\nPart 2 primary (recall@5 after rerank, graph - embedding, n = {n2}): {d[0]:+.1%} [{d[1]:+.1%}, {d[2]:+.1%}]"
      f" -> {verdict(d)}; half-width {(d[2] - d[1]) / 2:.1%}")
print("Threat (both parts): the test items were generated from the same graph / hierarchy the graph method uses,"
      "\nso the graph results are upper bounds. ToolRet not used (no data licence).")

RESULTS SUMMARY — Notebook 4.2: tool selection over a tool graph, batch run, gpt-6-luna (no reasoning)
Part 1, TaskBench (planner over 23-40 tools; graph = published type-compatible tool graph)
domain         metric                   all   embed   graph      graph - embed [95% CI]        graph - all [95% CI]
huggingface    node-F1 (primary)      87.0%   81.5%   79.7%        -1.8% [-3.2%, -0.5%]        -7.3% [-9.2%, -5.3%]
huggingface    edge-F1                50.0%   37.0%   38.1%        +1.1% [-2.3%, +4.6%]      -11.9% [-16.2%, -7.5%]
huggingface    exact node set         62.6%   51.6%   48.0%        -3.6% [-6.0%, -1.2%]     -14.6% [-18.4%, -11.0%]
huggingface    prompt tokens          1,439     679     978           +298 [+295, +302]           -462 [-465, -458]
multimedia     node-F1 (primary)      91.8%   87.0%   86.0%        -1.0% [-2.4%, +0.4%]        -5.8% [-7.9%, -3.8%]
multimedia     edge-F1                68.8%   54.5%   54.3%        -0.2% [-3.4%, +3.0%]     -14.5% [-18.8%, -1